# This chaper(codes) are only for learning purpose 

### READ_CSV() : 

In [ ]:
# Basic read — works for 80% of cases
df = pd.read_csv('employees.csv')

# filepath_or_buffer → path to file OR a URL directly
df = pd.read_csv('https://raw.githubusercontent.com/.../data.csv')

# sep → delimiter. Default is comma. For tab-separated use '\t'
df = pd.read_csv('data.tsv', sep='\t')

# header → which row is the header. Default=0 (first row)
# header=None → file has no header, Pandas assigns 0,1,2...
df = pd.read_csv('data.csv', header=None)

# names → provide custom column names
df = pd.read_csv('data.csv', header=None,
                  names=['ID', 'Name', 'Salary'])

# index_col → which column to use as the DataFrame index
df = pd.read_csv('data.csv', index_col='EmpID')

# usecols → read ONLY specific columns — saves memory on large files
df = pd.read_csv('data.csv', usecols=['Name', 'Salary', 'Dept'])

## READ_CSV() : Advance method

In [ ]:
# parse_dates → auto-convert date columns to datetime on read
# Saves you writing pd.to_datetime() separately
df = pd.read_csv('orders.csv', parse_dates=['OrderDate', 'DelivDate'])

# dtype → specify data types for columns manually
# WHY? Pandas type inference is not always correct
# EmpID might be read as int but you want it as string
# Salary might be read as float but it's always int
df = pd.read_csv('data.csv', dtype={
    'EmpID': str,
    'Salary': int,
    'Pincode': str   # pincodes like 011001 lose leading zero if read as int
})

# na_values → treat custom values as NaN
# Some datasets use 'N/A', 'NA', '-', 'null', 'none', '?' for missing
df = pd.read_csv('data.csv', na_values=['N/A', '-', 'null', '?', ''])

# skiprows → skip specific rows at the top
# Useful when file has metadata/comments before the actual data
df = pd.read_csv('data.csv', skiprows=3)   # skip first 3 rows

# nrows → read only first N rows — great for previewing huge files
df = pd.read_csv('huge_file.csv', nrows=1000)

# encoding → handle non-UTF-8 files (common with Indian language data)
df = pd.read_csv('data.csv', encoding='utf-8')    # default
df = pd.read_csv('data.csv', encoding='latin-1')   # for Windows-generated files

## READ_EXCEL()

In [ ]:
# read_excel() → reads .xlsx and .xls files
# Requires openpyxl library: pip install openpyxl

# Basic read — reads the FIRST sheet by default
df = pd.read_excel('report.xlsx')

# sheet_name → specify which sheet to read
df = pd.read_excel('report.xlsx', sheet_name='Sales')
df = pd.read_excel('report.xlsx', sheet_name=0)      # first sheet by index

# Read ALL sheets at once → returns a dictionary of DataFrames
all_sheets = pd.read_excel('report.xlsx', sheet_name=None)
# all_sheets['Sales'] → Sales sheet DataFrame
# all_sheets['HR']    → HR sheet DataFrame

# Most other parameters same as read_csv
# usecols, skiprows, nrows, dtype, na_values all work
df = pd.read_excel('report.xlsx',
                    sheet_name='Q1',
                    usecols='A:E',        # Excel column letters
                    skiprows=2,
                    nrows=500)

## TO_CSV() AND TO_EXCEL() : writing files

In [ ]:
# to_csv() → saves DataFrame to a CSV file

# Basic write
df.to_csv('output.csv')
# Problem: saves the DataFrame index as an extra column
# Open in Excel and you'll see an ugly unnamed column: 0,1,2,3

# Fix: always use index=False
df.to_csv('output.csv', index=False)

# encoding for special characters (Hindi names, accented chars)
df.to_csv('output.csv', index=False, encoding='utf-8-sig')
# utf-8-sig adds BOM marker → Excel opens UTF-8 files correctly

# to_excel() → saves to Excel file
df.to_excel('output.xlsx', index=False)

# sheet_name → name the sheet
df.to_excel('output.xlsx', sheet_name='CleanedData', index=False)

## Writing multiple sheets file to one excel

In [ ]:
# Writing multiple DataFrames to different sheets in ONE Excel file
# Use ExcelWriter as a context manager
# WHY context manager (with statement)?
# It ensures the file is properly saved and closed even if an error occurs

df_sales    = df[df['Dept'] == 'Sales']
df_hr       = df[df['Dept'] == 'HR']
df_summary  = df.groupby('Dept')['Salary'].mean().reset_index()

with pd.ExcelWriter('report.xlsx', engine='openpyxl') as writer:
    df_sales.to_excel(writer, sheet_name='Sales',   index=False)
    df_hr.to_excel(writer,    sheet_name='HR',      index=False)
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

# Result: report.xlsx with 3 sheets — Sales, HR, Summary
# This is exactly how you'd deliver a formatted report to a manager

## READ_JSON(): and chunking large files

In [ ]:
# read_json() → reads JSON files or JSON strings
df = pd.read_json('data.json')
df = pd.read_json('https://api.example.com/data')  # directly from API

# to_json() → saves to JSON
df.to_json('output.json', orient='records')
# orient='records' → list of dicts format [{col:val, ...}, ...]
# orient='columns' → dict of lists format {col: [val, val, ...]}

# ─────────────────────────────────────────────────────
# CHUNKING → reading huge files in pieces
# When file is too large to fit in RAM → read in chunks
# chunksize → number of rows per chunk

chunks = pd.read_csv('huge_file.csv', chunksize=10000)
# chunks is an iterator — not loaded into memory yet

results = []
for chunk in chunks:
    # process each chunk independently
    filtered = chunk[chunk['Salary'] > 50000]
    results.append(filtered)

final_df = pd.concat(results, ignore_index=True)

## Checking you file after reading

In [ ]:
# ALWAYS run these immediately after reading any file
# This is your standard "first look" checklist

df.shape           # (rows, columns) — how big is this data?
df.head()          # first 5 rows — does it look right?
df.dtypes          # are date columns datetime or object?
df.isnull().sum()  # how many missing values per column?
df.info()          # full summary — shape + dtypes + null counts
df.describe()      # statistical summary of numeric columns
df.columns.tolist() # exact column names (catch hidden spaces)